# AdaLi vs Sigmoid Comparison (MNIST, CIFAR-10, CIFAR-100)

This notebook compares **AdaLi** and **Sigmoid** surrogate activations from this repository using:
- Datasets: MNIST, CIFAR-10, CIFAR-100

In [ ]:
import os
import copy
import time
import random
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from config import AdaLiConfig
from modules.sgAdaLi import AdaLi
from modules.sgSigmoid import Sigmoid
from modules.neuron import LIF

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)

In [ ]:
# Reproducibility
SEED = 42

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('CUDA device 0:', torch.cuda.get_device_name(0))
else:
    print('CUDA is unavailable in this kernel.')
    if torch.version.cuda is None:
        print('Installed PyTorch build is CPU-only. Install a CUDA-enabled torch wheel in this environment to use the RTX 3050.')

# Keep a stable copy of the original AdaLi config values.
BASE_ADALI_CFG = copy.deepcopy(AdaLiConfig)
BASE_ADALI_CFG

In [ ]:
@dataclass
class ExperimentConfig:
    data_root: str = './data'
    batch_size: int = 256
    num_workers: int = 2
    persistent_workers: bool = True
    # Per-dataset epochs for fair comparison
    epochs_mnist: int = 50
    epochs_cifar10: int = 200
    epochs_cifar100: int = 300
    comparison_seeds: tuple[int, ...] = (42, 43, 44)
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # Quick mode: limit dataset size for faster prototyping.
    train_subset: int | None = 20000
    test_subset: int | None = 5000

    # SNN temporal parameters
    timesteps: int = 4

    adali_alpha: float = 0.5
    adali_beta: float = 0.5
    adali_left: float = 1.5
    adali_right: float = 1.5

    sigmoid_alpha: float = 4.0

cfg = ExperimentConfig()
cfg


In [ ]:
def _maybe_subset(dataset, n, seed=SEED):
    if n is None or n >= len(dataset):
        return dataset
    g = torch.Generator()
    g.manual_seed(seed)
    idx = torch.randperm(len(dataset), generator=g)[:n].tolist()
    return Subset(dataset, idx)


def _build_dataset(dataset_cls, root, train, transform, retries=3, base_wait_s=3):
    import urllib.error

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return dataset_cls(root, train=train, download=True, transform=transform)
        except (urllib.error.HTTPError, urllib.error.URLError) as err:
            last_err = err
            if attempt == retries:
                break
            wait_s = base_wait_s * attempt
            print(
                f"Download issue for {dataset_cls.__name__} (train={train}) - "
                f"attempt {attempt}/{retries}: {err}. Retrying in {wait_s}s..."
            )
            time.sleep(wait_s)

    # If files are already present locally, this succeeds without network.
    try:
        print(f"Using local cache for {dataset_cls.__name__} (train={train}).")
        return dataset_cls(root, train=train, download=False, transform=transform)
    except Exception as cache_err:
        raise RuntimeError(
            f"Failed to prepare {dataset_cls.__name__} (train={train}). "
            f"Last download error: {last_err}. Cache error: {cache_err}"
        ) from last_err


def get_dataloaders(dataset_name: str, cfg: ExperimentConfig):
    dataset_name = dataset_name.lower()

    if dataset_name == 'mnist':
        transform_train = transforms.Compose([
            transforms.RandomCrop(28, padding=2),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=False, transform=transform_test)
        in_channels = 1
        num_classes = 10

    elif dataset_name == 'cifar10':
        mean = (0.4914, 0.4822, 0.4465)
        std = (0.2470, 0.2435, 0.2616)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 10

    elif dataset_name == 'cifar100':
        mean = (0.5071, 0.4867, 0.4408)
        std = (0.2675, 0.2565, 0.2761)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 100

    else:
        raise ValueError(f'Unsupported dataset: {dataset_name}')

    train_set = _maybe_subset(train_set, cfg.train_subset)
    test_set = _maybe_subset(test_set, cfg.test_subset)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, test_loader, in_channels, num_classes

In [ ]:
from modules.neuron import LIF

def set_surrogate_mode(name: str, cfg: ExperimentConfig):
    # Reset global config first to avoid cross-run side effects.
    AdaLiConfig['Vth'] = BASE_ADALI_CFG['Vth']
    AdaLiConfig['Left'][0] = BASE_ADALI_CFG['Left'][0]
    AdaLiConfig['Right'][0] = BASE_ADALI_CFG['Right'][0]

    if name.lower() == 'adali':
        AdaLiConfig['Left'][0] = cfg.adali_left
        AdaLiConfig['Right'][0] = cfg.adali_right
    elif name.lower() == 'sigmoid':
        # Sigmoid constructor sets Left/Right to inf internally.
        pass
    else:
        raise ValueError(f'Unsupported surrogate: {name}')


def get_surrogate_function(name: str, cfg: ExperimentConfig):
    """Return a callable surrogate factory for use in neurons."""
    name = name.lower()

    if name == 'adali':
        class AdaLiSurrogate:
            def __init__(self):
                self.alpha = cfg.adali_alpha
                self.beta = cfg.adali_beta

            def __call__(self):
                return AdaLi(alpha=self.alpha, beta=self.beta)

        return AdaLiSurrogate()
    elif name == 'sigmoid':
        class SigmoidSurrogate:
            def __init__(self):
                self.alpha = cfg.sigmoid_alpha

            def __call__(self):
                return Sigmoid(alpha=self.alpha)

        return SigmoidSurrogate()
    else:
        raise ValueError(f'Unsupported surrogate: {name}')

In [ ]:
class SpikingSimpleCNN(nn.Module):
    """SNN with temporal spiking neurons using LIF neurons."""
    def __init__(self, in_channels: int, num_classes: int, surrogate_fn, timesteps: int = 4):
        super().__init__()
        self.timesteps = timesteps
        
        # Conv layer 1
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.neuron1 = LIF(surrogate_function=surrogate_fn)
        self.pool1 = nn.MaxPool2d(2)
        
        # Conv layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.neuron2 = LIF(surrogate_function=surrogate_fn)
        self.pool2 = nn.MaxPool2d(2)
        
        # Conv layer 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.neuron3 = LIF(surrogate_function=surrogate_fn)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # x shape: (batch, channels, height, width)
        batch_size = x.shape[0]
        outputs = []
        
        # Reset state once, then accumulate membrane dynamics across timesteps.
        self.neuron1.reset()
        self.neuron2.reset()
        self.neuron3.reset()
        
        # Process each timestep sequentially
        for t in range(self.timesteps):
            # Layer 1
            x_t = self.conv1(x)
            x_t = self.bn1(x_t)
            x_t = self.neuron1(x_t)
            x_t = self.pool1(x_t)
            
            # Layer 2
            x_t = self.conv2(x_t)
            x_t = self.bn2(x_t)
            x_t = self.neuron2(x_t)
            x_t = self.pool2(x_t)
            
            # Layer 3
            x_t = self.conv3(x_t)
            x_t = self.bn3(x_t)
            x_t = self.neuron3(x_t)
            x_t = self.pool3(x_t)
            
            # Classifier
            x_t = torch.flatten(x_t, 1)
            x_t = self.classifier(x_t)
            outputs.append(x_t)
        
        # Average outputs across timesteps
        x = torch.stack(outputs, dim=1).mean(dim=1)  # (batch, num_classes)
        return x


class SpikingBasicBlock(nn.Module):
    """Basic block for spiking ResNet."""
    def __init__(self, in_ch: int, out_ch: int, stride: int, surrogate_fn):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.neuron1 = LIF(surrogate_function=surrogate_fn)
        
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.neuron2 = LIF(surrogate_function=surrogate_fn)
        
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.neuron1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.neuron2(out)
        return out


class SpikingSmallResNet(nn.Module):
    """SNN ResNet with temporal spiking neurons."""
    def __init__(self, in_channels: int, num_classes: int, surrogate_fn, timesteps: int = 4):
        super().__init__()
        self.timesteps = timesteps
        
        self.stem_conv = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(32)
        self.stem_neuron = LIF(surrogate_function=surrogate_fn)
        
        self.layer1 = nn.Sequential(
            SpikingBasicBlock(32, 32, stride=1, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(32, 32, stride=1, surrogate_fn=surrogate_fn),
        )
        self.layer2 = nn.Sequential(
            SpikingBasicBlock(32, 64, stride=2, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(64, 64, stride=1, surrogate_fn=surrogate_fn),
        )
        self.layer3 = nn.Sequential(
            SpikingBasicBlock(64, 128, stride=2, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(128, 128, stride=1, surrogate_fn=surrogate_fn),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
    
    def _reset_neurons(self):
        """Recursively reset all LIF neurons."""
        for module in self.modules():
            if isinstance(module, LIF):
                module.reset()
    
    def forward(self, x):
        batch_size = x.shape[0]
        outputs = []
        
        # Reset state once, then accumulate membrane dynamics across timesteps.
        self._reset_neurons()
        
        # Process each timestep sequentially
        for t in range(self.timesteps):
            # Stem
            x_t = self.stem_conv(x)
            x_t = self.stem_bn(x_t)
            x_t = self.stem_neuron(x_t)
            
            # ResNet layers
            x_t = self.layer1(x_t)
            x_t = self.layer2(x_t)
            x_t = self.layer3(x_t)
            
            # Classification
            x_t = self.pool(x_t)
            x_t = torch.flatten(x_t, 1)
            x_t = self.fc(x_t)
            outputs.append(x_t)
        
        # Average outputs across timesteps
        x = torch.stack(outputs, dim=1).mean(dim=1)
        return x


def build_model(model_name: str, in_channels: int, num_classes: int, surrogate_fn, timesteps: int):
    name = model_name.lower()
    if name == 'simple_cnn':
        return SpikingSimpleCNN(in_channels=in_channels, num_classes=num_classes, 
                               surrogate_fn=surrogate_fn, timesteps=timesteps)
    if name == 'small_resnet':
        return SpikingSmallResNet(in_channels=in_channels, num_classes=num_classes, 
                                 surrogate_fn=surrogate_fn, timesteps=timesteps)
    raise ValueError(f'Unsupported model: {model_name}')

In [ ]:
def run_experiment(dataset_name: str, model_name: str, surrogate_name: str, cfg: ExperimentConfig, seed: int):
    set_global_seed(seed)
    set_surrogate_mode(surrogate_name, cfg)

    # Set per-dataset epochs
    if dataset_name == 'mnist':
        epochs = cfg.epochs_mnist
    elif dataset_name == 'cifar10':
        epochs = cfg.epochs_cifar10
    elif dataset_name == 'cifar100':
        epochs = cfg.epochs_cifar100
    else:
        epochs = cfg.epochs_mnist

    train_loader, test_loader, in_channels, num_classes = get_dataloaders(dataset_name, cfg)
    surrogate_fn = get_surrogate_function(surrogate_name, cfg)
    model = build_model(model_name, in_channels=in_channels, num_classes=num_classes, 
                       surrogate_fn=surrogate_fn, timesteps=cfg.timesteps).to(DEVICE)
    add_spike_count_to_model(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    history = []
    start_time = time.time()
    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        test_loss, test_acc, test_metrics = evaluate(model, test_loader, criterion, DEVICE)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            **{f'train_{k}': v for k, v in train_metrics.items()},
            **{f'test_{k}': v for k, v in test_metrics.items()},
        })
        print(f'[{dataset_name:8s}] [{model_name:11s}] [{surrogate_name:7s}] Epoch {epoch:3d}/{epochs} | '
              f'train_acc={train_acc:.4f} test_acc={test_acc:.4f}')
    elapsed = time.time() - start_time
    best_test_acc = max(h['test_acc'] for h in history)
    return {
        'dataset': dataset_name,
        'model': model_name,
        'surrogate': surrogate_name,
        'seed': seed,
        'epochs': epochs,
        'timesteps': cfg.timesteps,
        'best_test_acc': best_test_acc,
        'final_test_acc': history[-1]['test_acc'],
        'train_subset': cfg.train_subset,
        'test_subset': cfg.test_subset,
        'batch_size': cfg.batch_size,
        'budget_steps': epochs * len(train_loader),
        'seconds': elapsed,
        'history': history,
    }


In [ ]:

def add_spike_count_to_model(model):
    def spike_count():
        spikes = 0.0
        neurons = 0
        timesteps = 0
        for m in model.modules():
            if hasattr(m, 'spike_count'):
                s, n, t = m.spike_count()
                spikes += s
                neurons += n
                timesteps += t
        return spikes, neurons, timesteps
    model.spike_count = spike_count

# Patch LIF to count spikes
from modules.neuron import LIF
old_forward = LIF.forward

def lif_forward_with_spike_count(self, x):
    out = old_forward(self, x)
    if not hasattr(self, '_spike_count'):
        self._spike_count = 0.0
        self._spike_timesteps = 0
        self._spike_neurons = x.numel() // x.shape[0]  # neurons per sample
    self._spike_count += out.detach().sum().item()
    self._spike_timesteps += 1
    return out
LIF.forward = lif_forward_with_spike_count

def lif_spike_count(self):
    count = getattr(self, '_spike_count', 0.0)
    neurons = getattr(self, '_spike_neurons', 0)
    timesteps = getattr(self, '_spike_timesteps', 0)
    return count, neurons, timesteps
LIF.spike_count = lif_spike_count


In [ ]:
def _compute_snn_metrics(model):
    if not hasattr(model, 'spike_count'):
        return {
            'total_spikes': 0.0,
            'mean_spikes_per_neuron_per_timestep': 0.0,
        }

    spikes, neurons, timesteps = model.spike_count()
    denom = float(neurons * timesteps) if neurons > 0 and timesteps > 0 else 0.0
    mean_spikes = float(spikes / denom) if denom > 0 else 0.0
    return {
        'total_spikes': float(spikes),
        'mean_spikes_per_neuron_per_timestep': mean_spikes,
    }


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    loss = running_loss / max(total, 1)
    acc = correct / max(total, 1)
    metrics = _compute_snn_metrics(model)
    return loss, acc, metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    loss = running_loss / max(total, 1)
    acc = correct / max(total, 1)
    metrics = _compute_snn_metrics(model)
    return loss, acc, metrics

In [ ]:
# Reset experiments to retry with fixed architecture
all_runs = []
done = set()
failed_runs = []
print("Experiment state reset. Ready to run benchmark grid.")

In [ ]:
def run_benchmark_for_dataset(dataset_name: str, models: list[str] | None = None):
    dataset_name = dataset_name.lower()
    models = [m.lower() for m in models] if models is not None else ['simple_cnn', 'small_resnet']
    surrogates = ['adali', 'sigmoid']
    seeds = tuple(cfg.comparison_seeds)

    global all_runs, done, failed_runs

    if 'all_runs' not in globals() or not isinstance(all_runs, list):
        all_runs = []

    done = {(r['dataset'], r['model'], r['surrogate'], r.get('seed', SEED)) for r in all_runs}
    failed_runs = []

    try:
        _ = get_dataloaders(dataset_name, cfg)
        print(f'Dataset ready: {dataset_name}')
    except Exception as err:
        print(f'Skipping dataset {dataset_name} (data unavailable): {err}')
        return []

    new_results = []
    for model_name in models:
        for seed in seeds:
            for surr in surrogates:
                key = (dataset_name, model_name, surr, seed)
                if key in done:
                    print(f'Skipping already completed: {key}')
                    continue

                try:
                    result = run_experiment(dataset_name, model_name, surr, cfg, seed=seed)
                    all_runs.append(result)
                    new_results.append(result)
                    done.add(key)
                except Exception as err:
                    msg = str(err)
                    failed_runs.append({'dataset': dataset_name, 'model': model_name, 'surrogate': surr, 'seed': seed, 'error': msg})
                    print(f'Failed {key}: {msg}')

    print(f'New runs completed for {dataset_name}: {len(new_results)}')
    if failed_runs:
        print(f'Failed runs in this pass: {len(failed_runs)}')
        for fr in failed_runs:
            print(f"- ({fr['dataset']}, {fr['model']}, {fr['surrogate']})")

    return new_results


## Run Benchmarks Per Dataset

Run any of the following cells independently. They append into `all_runs`, so you can execute only the datasets you want and then build the summary tables from the accumulated results.

In [ ]:
# LOCAL (laptop) - simple_cnn only
# Produces a portable artifact you can keep or upload to Colab.
run_benchmark_for_dataset('mnist',    models=['simple_cnn'])
run_benchmark_for_dataset('cifar10',  models=['simple_cnn'])
run_benchmark_for_dataset('cifar100', models=['simple_cnn'])

import pickle
local_out = 'all_runs_simple_cnn_laptop.pkl'
pickle.dump(all_runs, open(local_out, 'wb'))
print(f'Saved {len(all_runs)} local runs to {local_out}')

In [ ]:
# COLAB - small_resnet only
# Upload all_runs_simple_cnn_laptop.pkl to Colab if you want one merged artifact there.
run_benchmark_for_dataset('mnist',    models=['small_resnet'])
run_benchmark_for_dataset('cifar10',  models=['small_resnet'])
run_benchmark_for_dataset('cifar100', models=['small_resnet'])

# Save results for transfer back to laptop
import pickle
colab_out = 'all_runs_small_resnet_colab.pkl'
pickle.dump(all_runs, open(colab_out, 'wb'))
print(f'Saved {len(all_runs)} Colab runs to {colab_out}')

In [ ]:
# Merge local + colab artifacts (if present), then build summary tables.
import os
import pickle

candidate_files = [
    'all_runs_simple_cnn_laptop.pkl',
    'all_runs_small_resnet_colab.pkl',
    'all_runs_resnet.pkl',  # backward compatibility with older filename
]

merged_runs = []
seen = set()

for fp in candidate_files:
    if not os.path.exists(fp):
        continue
    loaded = pickle.load(open(fp, 'rb'))
    for r in loaded:
        key = (r.get('dataset'), r.get('model'), r.get('surrogate'), r.get('seed', SEED))
        if key in seen:
            continue
        seen.add(key)
        merged_runs.append(r)

# Fallback to in-memory runs if no artifact files exist.
if not merged_runs:
    merged_runs = list(all_runs)

# Keep notebook state consistent for downstream cells.
all_runs = merged_runs
print(f'Total merged runs: {len(all_runs)}')

records = []
for r in all_runs:
    rec = {
        'dataset': r['dataset'],
        'model': r['model'],
        'surrogate': r['surrogate'],
        'seed': r.get('seed', SEED),
        'best_test_acc': r['best_test_acc'],
        'final_test_acc': r['final_test_acc'],
        'seconds': r['seconds'],
        'epochs': r['epochs'],
        'batch_size': r.get('batch_size', cfg.batch_size),
        'budget_steps': r.get('budget_steps', np.nan),
        'train_subset': r['train_subset'],
        'test_subset': r['test_subset'],
    }
    # Add SNN metrics if present.
    if 'history' in r and isinstance(r['history'], list) and len(r['history']) > 0:
        for k in r['history'][-1]:
            if k.startswith('test_') or k.startswith('train_'):
                rec[k] = r['history'][-1][k]
    records.append(rec)

df = pd.DataFrame(records).sort_values(['dataset', 'model', 'surrogate', 'seed']).reset_index(drop=True)
summary = (
    df.groupby(['dataset', 'model', 'surrogate'], as_index=False)
      .agg(
          n_seeds=('seed', 'nunique'),
          best_test_acc_mean=('best_test_acc', 'mean'),
          best_test_acc_std=('best_test_acc', 'std'),
          final_test_acc_mean=('final_test_acc', 'mean'),
          final_test_acc_std=('final_test_acc', 'std'),
          seconds_mean=('seconds', 'mean'),
          test_mean_spikes_per_neuron_per_timestep=('test_mean_spikes_per_neuron_per_timestep', 'mean'),
          test_total_spikes=('test_total_spikes', 'mean'),
      )
)
summary

In [ ]:
pivot = summary.pivot_table(
    index=['dataset', 'model'],
    columns='surrogate',
    values='best_test_acc_mean',
    aggfunc='max'
).reset_index()

if {'adali', 'sigmoid'}.issubset(set(pivot.columns)):
    pivot['adali_minus_sigmoid'] = pivot['adali'] - pivot['sigmoid']

pivot

In [ ]:
# Visual comparison
plot_df = summary.copy()
plot_df['label'] = plot_df['dataset'] + ' | ' + plot_df['model']

plt.figure(figsize=(12, 5))
for surrogate_name, grp in plot_df.groupby('surrogate'):
    x = np.arange(len(grp))

# Keep deterministic order across groups
plot_df = plot_df.sort_values(['dataset', 'model', 'surrogate']).reset_index(drop=True)
labels = sorted(plot_df['label'].unique())
x = np.arange(len(labels))
width = 0.35

vals_adali = []
vals_sigmoid = []
for lbl in labels:
    vals_adali.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'adali')]['best_test_acc_mean'].iloc[0]))
    vals_sigmoid.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'sigmoid')]['best_test_acc_mean'].iloc[0]))

plt.bar(x - width / 2, vals_adali, width=width, label='AdaLi')
plt.bar(x + width / 2, vals_sigmoid, width=width, label='Sigmoid')
plt.xticks(x, labels, rotation=30, ha='right')
plt.ylabel('Best Test Accuracy')
plt.title('AdaLi vs Sigmoid across Dataset/Architecture')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save merged artifacts for reporting/reproducibility
import pickle

out_csv = 'adali_vs_sigmoid_results.csv'
out_runs = 'all_runs_merged.pkl'
out_summary = 'adali_vs_sigmoid_summary.csv'

df.to_csv(out_csv, index=False)
summary.to_csv(out_summary, index=False)
pickle.dump(all_runs, open(out_runs, 'wb'))

print('Saved:', out_csv)
print('Saved:', out_summary)
print('Saved:', out_runs)